# 00: Local smoke test with Qiskit SDK reference primitives

This notebook proves a **no-account, no-QPU** local path using Qiskit SDK reference primitives.

What this demonstrates:
- Build a tiny Bell circuit.
- Run it with `StatevectorSampler` locally.
- Print bitstrings and counts as a quick reproducibility check.

In [ ]:
import numpy as np
import qiskit
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector

print('Qiskit version:', qiskit.__version__)

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# Check the state before adding measurements: (|00> + |11>) / sqrt(2).
state = Statevector.from_instruction(qc)
np.testing.assert_allclose(state.data, np.array([1, 0, 0, 1]) / np.sqrt(2), rtol=0, atol=1e-12)
np.testing.assert_allclose(state.probabilities(), [0.5, 0, 0, 0.5], rtol=0, atol=1e-12)
np.testing.assert_allclose(state.expectation_value(SparsePauliOp('ZZ')), 1.0, rtol=0, atol=1e-12)
qc.measure_all()

qc.draw('text')

In [ ]:
sampler = StatevectorSampler(seed=1234)
job = sampler.run([qc], shots=256)
pub_result = job.result()[0]

meas_data = pub_result.data["meas"]
counts = meas_data.get_counts()
bitstrings = meas_data.get_bitstrings()

print('Counts:', counts)
print('First 12 bitstrings:', bitstrings[:12])

assert sum(counts.values()) == 256
assert len(bitstrings) == 256
assert set(counts) == {'00', '11'}
# Allow six binomial standard deviations, not an exact random count.
assert abs(counts['00'] - 128) <= 6 * np.sqrt(256 * 0.25)
print('PASSED: exact Bell state and reference SamplerV2 counts')

Expected behavior: all 256 shots are split between `00` and `11`, approximately half each. The assertions check the exact Bell state and sampled outcomes; a red traceback is a failure.